# CottonLens — Research Workbench

Kalıcı araştırma defteri. Varsayılan `status` yalnız kayıt durumunu gösterir. `review`, önceki tahminleri ve veri kanıtını inceler; model eğitmez. `pilot_plan` doğrulanmış AMS/FAS paketleriyle iş bütçesini hesaplar. `pilot`, açık seçimle ve süre sınırıyla T+5 karşılaştırmasını çalıştırır. Tamamlanan büyük market ablation yeniden açılmaz.


In [ ]:
# Choose one workflow. Run-all follows this choice; default does not fit models.
WORKFLOW = 'status'  # status, review, pilot_plan, pilot
MAX_MINUTES = 60.  # Checked before each new fit; the active fit can finish later.
RUN_TRAINING = False  # Set True only when deliberately selecting pilot.
RECOVER_INTERRUPTED_WRITER = False
WORKFLOWS = {
    'status': [('status', 'A')],
    'review': [('review', 'A')],
    'pilot_plan': [('pilot-plan', 'A')],
    'pilot': [('diagnose', 'A'), ('pilot', 'A')],
}
assert WORKFLOW in WORKFLOWS
assert 0 < MAX_MINUTES <= 120
STAGES = WORKFLOWS[WORKFLOW]
print('Selected workflow:', WORKFLOW, '| maximum session minutes:', MAX_MINUTES)


In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/CottonLensAI')
REFERENCE_EXPERIMENT = 'research-market-v1'
REFERENCE_SOURCE_ID = '9e071564c70150ac1f25e6e506daac768ae58ad556ddca38c93c4b16cf73e0c4'
EXPERIMENT = 'research-information-pilot-v1'  # Separate from the completed market experiment.
PUBLICATION_PACKAGES = []  # Verified as-published package folders only; raw archives are not packages.
EVIDENCE_ROOT = DRIVE_ROOT / 'reports' / 'research-readiness-evidence-v1'
READINESS = EVIDENCE_ROOT / 'output/reports/free-source-readiness-20261001-v3.json'


In [ ]:
import hashlib, importlib, json, sys, zipfile
from pathlib import PurePosixPath
SOURCE_ZIP = DRIVE_ROOT / 'sources/cottonlens-research-review-source-v1.zip'
EXPECTED_SOURCE_ID = 'fab20e76d17bc7f7a07b016be63a7c821a92cffa58942c9d0b379a02143ee38d'
EXPECTED_ZIP_SHA256 = '4291de932f5101f1c50b5cfa90b6f262298e1f0c7813217eebcf2ed346beb17d'
expected_zip = SOURCE_ZIP.with_suffix('.zip.sha256').read_text().split()[0]
assert expected_zip == EXPECTED_ZIP_SHA256, 'Source ZIP sidecar differs from pinned bundle'
assert hashlib.sha256(SOURCE_ZIP.read_bytes()).hexdigest() == EXPECTED_ZIP_SHA256, 'Source ZIP checksum mismatch'
with zipfile.ZipFile(SOURCE_ZIP) as archive:
    source = json.loads(archive.read('.source-manifest.json'))
    payload = {key: value for key, value in source.items() if key != 'source_id'}
    identity = hashlib.sha256(json.dumps(payload, sort_keys=True, separators=(',', ':')).encode()).hexdigest()
    assert identity == source['source_id'] == EXPECTED_SOURCE_ID, 'Source manifest mismatch'
    assert len(archive.namelist()) == len(set(archive.namelist())), 'Duplicate archive members'
    assert set(archive.namelist()) == set(source['files']) | {'.source-manifest.json'}, 'Unexpected archive files'
    REPO = Path('/content') / ('cottonlens-source-' + identity[:16])
    REPO.mkdir(exist_ok=True)
    for name in archive.namelist():
        member = PurePosixPath(name)
        assert not member.is_absolute() and '..' not in member.parts and chr(92) not in name and ':' not in name
        content = archive.read(name)
        if name in source['files']:
            assert hashlib.sha256(content).hexdigest() == source['files'][name], name
        target = REPO / name
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists():
            assert target.read_bytes() == content, f'Existing Colab source changed: {name}'
        else:
            with target.open('xb') as handle:
                handle.write(content)
sys.path.insert(0, str(REPO / 'ml'))
import colab_setup
importlib.reload(colab_setup)
colab_setup.PROJECT = REPO
from colab_setup import setup, run, INFERENCE
print('Source identity:', identity, 'Git HEAD:', source['git_head'], 'Includes working-tree edits:', source['working_tree_dirty'])


In [ ]:
PYTHON = setup(research=True)  # Locked managed Python 3.12.11; no global packages.
print('Training interpreter:', PYTHON)
print('Inference-only interpreter:', INFERENCE / 'bin/python')


In [ ]:
import json
from colab_setup import process_env
reference = DRIVE_ROOT / 'experiments' / REFERENCE_EXPERIMENT
reference_ready = json.loads((reference / 'ready.json').read_text(encoding='utf-8'))
assert reference_ready['identity']['source_id'] == REFERENCE_SOURCE_ID
assert reference_ready['identity']['publication_sources'] == []
assert reference_ready['identity']['source_data']['data_id'] == 'b7aecc5c96a240069a3251e3e88b5f1c0c448fac5c8eb2f02782a16b4a0c03ca'
assert sum(len(f['origins']) for f in reference_ready['identity']['split']['folds']) == 1008
REFERENCE_BASE = [PYTHON, '-m', 'cottonlens_ml.research.engine', '--repo', REPO,
                  '--drive-root', DRIVE_ROOT, '--experiment', REFERENCE_EXPERIMENT]
folder = DRIVE_ROOT / 'experiments' / EXPERIMENT
BASE = [PYTHON, '-m', 'cottonlens_ml.research.engine', '--repo', REPO,
        '--drive-root', DRIVE_ROOT, '--experiment', EXPERIMENT]
PILOT_READY = False
if WORKFLOW in ('pilot_plan', 'pilot'):
    if WORKFLOW == 'pilot':
        assert RUN_TRAINING, 'Set RUN_TRAINING=True only for an intentional Colab pilot.'
    if not (folder / 'ready.json').is_file() and not PUBLICATION_PACKAGES:
        print('BLOCKED: no verified AMS/FAS publication package. Use review; do not rerun the completed ablation.')
    else:
        lock = folder / '.writer-lock'
        if lock.exists():
            import colab_progress
            owner = json.loads((lock / 'owner.json').read_text(encoding='utf-8'))
            assert RECOVER_INTERRUPTED_WRITER, 'Stop/delete the previous runtime before explicit lock recovery.'
            colab_progress.recover_owned_lock(folder, owner_token=owner['token'], confirmed_stopped=True)
        if not (folder / 'ready.json').is_file():
            package_args = [part for p in PUBLICATION_PACKAGES for part in ('--publication-package', str(p))]
            run([*BASE, '--stage', 'prepare', '--profile', 'free-data-v1', *package_args])
        ready = json.loads((folder / 'ready.json').read_text(encoding='utf-8'))
        assert ready['identity']['source_id'] == identity, 'Pilot source changed; preserve it and use a new namespace.'
        assert ready['identity']['publication_sources'], 'Market-only experiments cannot enter the new-information pilot.'
        assert ready['identity']['profile'] == 'free-data-v1'
        PILOT_READY = True
print('Historical reference preserved. Reviewer source and historical training source have separate identities.')


In [ ]:
import colab_progress
from IPython.display import display, Markdown
from colab_setup import process_env

if WORKFLOW == 'status':
    run([*REFERENCE_BASE, '--stage', 'status'])
elif WORKFLOW == 'review':
    run([PYTHON, '-m', 'cottonlens_ml.research.review', '--experiment-root', reference,
         '--evidence-root', EVIDENCE_ROOT, '--readiness', READINESS, '--output', DRIVE_ROOT / 'reports'])
elif PILOT_READY:
    for stage, round_name in STAGES:
        command = [*BASE, '--stage', stage, '--round', round_name]
        if stage == 'pilot-plan':
            run(command)
            continue
        if stage == 'diagnose' and (folder / 'diagnosis.json').exists():
            assert json.loads((folder / 'diagnosis.json').read_text())['status'] == 'passed'
            print('Diagnosis already passed; controls will not be repeated.')
            continue
        candidate_total = 8
        if stage == 'pilot':
            import subprocess
            budget = json.loads(subprocess.check_output([str(p) for p in [*BASE, '--stage', 'pilot-plan']],
                                                        cwd=REPO, env=process_env(), text=True))
            print('Pilot budget:', budget)
            assert budget['status'] == 'ready', budget.get('blocker')
            candidate_total = budget['candidate_count']
            command += ['--max-minutes', str(MAX_MINUTES)]
        panel = display(Markdown('Starting/resuming ' + stage), display_id=True)
        def update(message):
            panel.update(Markdown('```text\n' + message + '\n```'))
        colab_progress.supervise(command, cwd=REPO, env=process_env(),
            drive_logs=DRIVE_ROOT / 'reports', decisions=folder / ('ledger/completed' if stage == 'diagnose' else 'pilot/candidates'),
            decision_pattern='*.json', total=candidate_total,
            fold_count=len(ready['identity']['split']['folds']), update=update)
    run([*BASE, '--stage', 'status'])
else:
    print('No model training was started. Source admission is still required.')


In [ ]:
# Read only compact, immutable review/pilot summaries.
reviews = sorted((DRIVE_ROOT / 'reports').glob('research-review-*/review.json'), key=lambda p: p.stat().st_mtime)
if reviews:
    review = json.loads(reviews[-1].read_text(encoding='utf-8'))
    print('Review:', reviews[-1], '| new fits:', review['new_fits'])
    print('Source pilot:', review['source_inventory']['pilot_status'])
summary_path = folder / 'pilot/summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print('Pilot:', summary['status'], '| candidates:', summary['saved_candidates'])
    print('Short pilot is research prioritization; it cannot pass the eight-fold release gate.')
elif WORKFLOW == 'pilot' and PILOT_READY:
    print('Pilot paused or incomplete. Resume the same source and experiment; completed fits remain cached.')
print('Next: review source eligibility and error strata before enlarged search.')
print('ML term — MAE: average absolute error in the original price unit.')
